# DrivenData ASR Training Runner

Watches Google Drive for trigger configs from GitHub Actions.
On detection: installs deps, downloads data, runs train.py, uploads results.

**Usage**: Run this notebook on Colab with GPU runtime. RPi5 keepalive keeps the session alive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
# Base folder on Drive (shared with Service Account)
DRIVE_BASE = '/content/drive/MyDrive/kaggle/drivendata-pasketti/experiments'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive base: {DRIVE_BASE}')
print(f'Existing experiments: {os.listdir(DRIVE_BASE) if os.path.exists(DRIVE_BASE) else "none"}')

In [ ]:
# Core utilities
import json
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path


def log(msg):
    """Timestamped log."""
    ts = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    print(f'[{ts}] {msg}')


def find_pending_experiments():
    """Scan Drive for experiments with status=pending."""
    pending = []
    base = Path(DRIVE_BASE)
    if not base.exists():
        return pending

    for exp_dir in sorted(base.iterdir()):
        trigger = exp_dir / 'trigger.json'
        if trigger.exists():
            try:
                config = json.loads(trigger.read_text())
                if config.get('status') == 'pending':
                    pending.append((exp_dir, config))
            except Exception as e:
                log(f'Error reading {trigger}: {e}')
    return pending


def write_status(exp_dir, status, **extra):
    """Update trigger.json status."""
    trigger = exp_dir / 'trigger.json'
    config = json.loads(trigger.read_text())
    config['status'] = status
    config.update(extra)
    trigger.write_text(json.dumps(config, indent=2))
    log(f'Status -> {status}')


def write_result(exp_dir, status, metrics=None, error=None):
    """Write result.json for GitHub Actions to detect."""
    result = {
        'status': status,
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'metrics': metrics or {},
        'error': error,
    }
    (exp_dir / 'result.json').write_text(json.dumps(result, indent=2))
    log(f'Result written: {status}')


print('Utilities loaded.')

In [ ]:
def install_dependencies(exp_dir):
    """Install training dependencies."""
    log('Installing dependencies...')
    req_file = exp_dir / 'requirements-train.txt'
    if req_file.exists():
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)],
            check=True,
        )
    else:
        # Fallback: install common ASR dependencies
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'torch', 'torchaudio', 'transformers[torch]>=4.52.4',
             'datasets', 'librosa', 'soundfile', 'jiwer', 'wandb', 'requests'],
            check=True,
        )
    log('Dependencies installed.')


def download_data(config, work_dir):
    """Download competition data from GitHub Artifact."""
    data_config = config['data']
    source = data_config.get('source', 'github-artifact')

    if source != 'github-artifact':
        raise ValueError(f'Unknown data source: {source}')

    log(f'Downloading data from GitHub Artifact: {data_config["artifact_name"]}')

    import io
    import zipfile
    import requests

    gh_pat = data_config['gh_pat']
    repo = data_config['repo']
    artifact_name = data_config['artifact_name']

    headers = {
        'Authorization': f'token {gh_pat}',
        'Accept': 'application/vnd.github+json',
    }

    # Find latest artifact
    resp = requests.get(
        f'https://api.github.com/repos/{repo}/actions/artifacts',
        headers=headers,
        params={'name': artifact_name, 'per_page': 1},
    )
    resp.raise_for_status()
    artifacts = resp.json()['artifacts']
    if not artifacts:
        raise RuntimeError(f'Artifact "{artifact_name}" not found in {repo}')

    artifact = artifacts[0]
    if artifact.get('expired'):
        raise RuntimeError(f'Artifact "{artifact_name}" expired. Re-run Download Competition Data.')

    size_mb = artifact['size_in_bytes'] / 1024 / 1024
    log(f'Downloading: {artifact_name} ({size_mb:.1f} MB)')

    resp = requests.get(artifact['archive_download_url'], headers=headers)
    resp.raise_for_status()

    data_dir = work_dir / config['train_args']['data_dir']
    data_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        zf.extractall(data_dir)

    files = list(data_dir.iterdir())
    log(f'Extracted {len(files)} files to {data_dir}')
    for f in sorted(files)[:10]:
        log(f'  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)')

    return data_dir


print('Data functions loaded.')

In [ ]:
def run_training(config, exp_dir, work_dir):
    """Execute train.py with configured arguments."""
    import torch

    log(f'GPU available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        log(f'GPU: {torch.cuda.get_device_name(0)}')
        log(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

    train_args = config['train_args']
    train_py = exp_dir / 'train.py'

    # Copy train.py to work_dir
    import shutil
    work_train_py = work_dir / 'train.py'
    shutil.copy2(train_py, work_train_py)

    # Set up W&B
    wandb_key = config.get('wandb_api_key', '')
    env = os.environ.copy()
    if wandb_key:
        env['WANDB_API_KEY'] = wandb_key
        env['WANDB_MODE'] = 'offline'  # Save locally, sync later

    # Build command
    cmd = [
        sys.executable, str(work_train_py),
        '--data_dir', str(work_dir / train_args['data_dir']),
        '--output_dir', str(work_dir / train_args['output_dir']),
        '--model_name', train_args['model_name'],
        '--epochs', str(train_args['epochs']),
        '--batch_size', str(train_args['batch_size']),
        '--gradient_accumulation', str(train_args['gradient_accumulation']),
        '--lr', str(train_args['lr']),
        '--wandb_project', train_args.get('wandb_project', 'drivendata-phonetic-asr'),
        '--memo', train_args.get('memo', 'colab'),
    ]

    log(f'Running: {" ".join(cmd[:4])} ...')
    log(f'Args: model={train_args["model_name"]} epochs={train_args["epochs"]} bs={train_args["batch_size"]} lr={train_args["lr"]}')

    result = subprocess.run(
        cmd,
        env=env,
        cwd=str(work_dir),
        capture_output=False,  # Stream output to notebook
    )

    if result.returncode != 0:
        raise RuntimeError(f'train.py exited with code {result.returncode}')

    log('Training completed successfully.')
    return work_dir / train_args['output_dir']


print('Training function loaded.')

In [ ]:
def package_and_upload_results(config, exp_dir, output_dir):
    """Package model as tar.gz and upload to Drive experiment folder."""
    import tarfile
    import glob
    import shutil

    model_dir = output_dir / 'final_model'
    if not model_dir.exists():
        raise FileNotFoundError(f'Model directory not found: {model_dir}')

    # Create model tar.gz
    tar_path = exp_dir / 'model.tar.gz'
    log(f'Packaging model from {model_dir}...')
    with tarfile.open(tar_path, 'w:gz') as tar:
        tar.add(model_dir, arcname='.')

    size_mb = tar_path.stat().st_size / 1024 / 1024
    log(f'Model archive: {tar_path.name} ({size_mb:.1f} MB)')

    # Package W&B offline runs if they exist
    wandb_dirs = list(output_dir.rglob('offline-run-*'))
    if not wandb_dirs:
        wandb_dirs = list(Path('/content').rglob('wandb/offline-run-*'))

    if wandb_dirs:
        wandb_tar = exp_dir / f'wandb-offline-{wandb_dirs[0].name}.tar.gz'
        with tarfile.open(wandb_tar, 'w:gz') as tar:
            for d in wandb_dirs:
                tar.add(d, arcname=d.name)
        log(f'W&B offline runs packaged: {wandb_tar.name}')
    else:
        log('No W&B offline runs found.')

    return tar_path


def extract_metrics(output_dir):
    """Extract final metrics from training output."""
    metrics = {}

    # Check trainer_state.json for best metric
    trainer_state = output_dir / 'checkpoints' / 'trainer_state.json'
    if trainer_state.exists():
        try:
            state = json.loads(trainer_state.read_text())
            best_metric = state.get('best_metric')
            if best_metric is not None:
                metrics['eval_cer'] = best_metric
                log(f'Best CER from trainer_state: {best_metric}')
        except Exception:
            pass

    # Check W&B summary
    try:
        import wandb
        if wandb.run:
            summary = dict(wandb.run.summary)
            if 'eval_cer' in summary:
                metrics['eval_cer'] = summary['eval_cer']
    except Exception:
        pass

    return metrics


print('Packaging functions loaded.')

In [ ]:
def run_experiment(exp_dir, config):
    """Execute a single experiment end-to-end."""
    exp_name = exp_dir.name
    log(f'=== Starting experiment: {exp_name} ===')
    log(f'Memo: {config.get("memo", "N/A")}')

    # Mark as running
    write_status(exp_dir, 'running')

    # Work directory (local Colab disk, fast I/O)
    work_dir = Path(f'/content/work/{exp_name}')
    work_dir.mkdir(parents=True, exist_ok=True)

    try:
        # Step 1: Install dependencies
        install_dependencies(exp_dir)

        # Step 2: Download data
        download_data(config, work_dir)

        # Step 3: Train
        output_dir = run_training(config, exp_dir, work_dir)

        # Step 4: Extract metrics
        metrics = extract_metrics(output_dir)

        # Step 5: Package and upload to Drive
        package_and_upload_results(config, exp_dir, output_dir)

        # Step 6: Write success result
        write_result(exp_dir, 'success', metrics=metrics)
        write_status(exp_dir, 'completed')
        log(f'=== Experiment {exp_name} COMPLETED ===')
        log(f'Metrics: {metrics}')

    except Exception as e:
        error_msg = f'{type(e).__name__}: {e}'
        log(f'=== Experiment {exp_name} FAILED: {error_msg} ===')
        traceback.print_exc()
        write_result(exp_dir, 'failed', error=error_msg)
        write_status(exp_dir, 'failed', error=error_msg)

    finally:
        # Clean up work directory to free disk
        import shutil
        if work_dir.exists():
            shutil.rmtree(work_dir, ignore_errors=True)
            log(f'Cleaned up: {work_dir}')


print('Experiment runner loaded.')

In [ ]:
# === Main Loop: Watch for pending experiments ===
# This cell runs indefinitely. RPi5 keepalive prevents Colab timeout.

POLL_INTERVAL = 60  # seconds between Drive scans

log('=== DrivenData ASR Runner Started ===')
log(f'Watching: {DRIVE_BASE}')
log(f'Poll interval: {POLL_INTERVAL}s')

cycle = 0
while True:
    cycle += 1
    try:
        pending = find_pending_experiments()
        if pending:
            log(f'Found {len(pending)} pending experiment(s)')
            for exp_dir, config in pending:
                run_experiment(exp_dir, config)
        elif cycle % 30 == 0:  # Log heartbeat every 30 minutes
            log(f'Heartbeat: no pending experiments (cycle {cycle})')
    except Exception as e:
        log(f'Runner error: {e}')
        traceback.print_exc()

    time.sleep(POLL_INTERVAL)